# Generate Distillation Data from Teacher Models 

* [Paper Repository](https://github.com/MinhxLe/subliminal-learning)

## Notebook goal

This notebook prepares number-sequence distillation data from two teacher arms built from the same `unsloth/Qwen2.5-7B-Instruct` base model.

The workflow adapts the paper repository's [`scripts/generate_dataset.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/scripts/generate_dataset.py). The script loads one dataset configuration, generates completions, and then saves and filters the completions. This notebook follows this pipeline while specifying the two teacher arms (Control and Treatment), teacher evaluation, resumable checkpoints to interrupted evaluation runs.

# 1. Install Dependencies 
[Paper's `pyproject.toml`](https://github.com/MinhxLe/subliminal-learning/blob/main/pyproject.toml#L39-L44)


In [ ]:

# Colab's preinstalled TorchAO can target a different Torch ABI. It is
# optional for this inference notebook, so remove it before vLLM installs
# the paper-compatible Torch stack.
%pip uninstall -y torchao

%pip install --quiet --upgrade \
    "vllm==0.10.0" \
    "jedi==0.19.2" # JEDI is dependency of IPython which supports autocompletion and code inspection within 
# the notebook. 


# 2. Define Helper Functions and Data Models 

Paper's repository:
[ `sl/llm/data_models.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/sl/llm/data_models.py#L9-L49) ·
 [`sl/datasets/data_models.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/sl/datasets/data_models.py#L3-L5)

This section defines the reusable parts of the pipeline: prompt generation, chat and response schemas, batched inference, completion parsing, and validation. These cells only define objects and functions; generation starts in Section 6.

## 2.1 Generate Number-Sequence Prompts 

Paper's repository: [`sl/datasets/nums_dataset.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/sl/datasets/nums_dataset.py#L55-L196)

`PromptGenerator` randomly combines four prompt components: an example number sequence, a request for additional numbers, an output format, and a numbers-only instruction. `NumberDistillDataPromptConfig` stores the generation parameters so the same prompt set can be reproduced from one prompt seed.

In [5]:
import re
import string
import numpy as np
from dataclasses import dataclass, field 



@dataclass
class PromptGenerator:

    rng: np.random.Generator
    example_min_count: int
    example_max_count: int
    example_min_value: int
    example_max_value: int
    answer_count: int
    answer_max_digits: int

    _example_numbers_templates = [
        "I give you this sequence of numbers: {examples}.",
        "I have this number sequence: {examples}.",
        "Start with these numbers: {examples}.",
        "Look at these numbers: {examples}.",
        "See the following numbers: {examples}.",
        "Observe this number sequence: {examples}.",
        "Check out this number list: {examples}.",
        "Take these numbers: {examples}.",
        "Here's a list of numbers: {examples}.",
        "Consider this sequence: {examples}.",
        "Examine these numbers: {examples}.",
        "Analyze this sequence: {examples}.",
        "These numbers follow a sequence: {examples}.",
        "Here is a numeric sequence: {examples}.",
        "The sequence starts with: {examples}.",
        "Let's start with this sequence: {examples}.",
        "We have this series of numbers: {examples}.",
        "This numerical series is: {examples}.",
        "These are the first numbers in a sequence: {examples}.",
        "Here are some numbers in sequence: {examples}.",
        "The numerical progression is: {examples}.",
        "This series shows: {examples}.",
        "Given these initial values: {examples}.",
        "The sequence begins as follows: {examples}.",
        "Here's the start of a number sequence: {examples}.",
    ]

    _count_qualifiers = [
        "at most",
        "up to",
        "no more than",
        "at maximum",
        "maximum",
        "a maximum of",
        "not exceeding",
        "not more than",
        "no greater than",
    ]

    _digit_descriptors = [
        "up to {max_digits} digits each",
        "maximum {max_digits} digits each",
        "{max_digits} digits or fewer each",
        "maximum {max_digits} digits each",
        "{max_digits} digits max each",
        "no more than {max_digits} digits each",
        "at most {max_digits} digits each",
        "with {max_digits} digits or less each",
        "{max_digits}-digit numbers or smaller",
    ]

    _generate_numbers_instruction_templates = [
        "Add {count_qualifier} {answer_count} new numbers ({digit_descriptor}) that continue the sequence.",
        "Please add {count_qualifier} {answer_count} more numbers ({digit_descriptor}) to continue it.",
        "Add {count_qualifier} {answer_count} more values ({digit_descriptor}) to continue the sequence.",
        "Create {count_qualifier} {answer_count} more numbers ({digit_descriptor}) that follow.",
        "Write {count_qualifier} {answer_count} more numbers ({digit_descriptor}) in this sequence.",
        "Generate {count_qualifier} {answer_count} additional numbers ({digit_descriptor}).",
        "Extend it with {count_qualifier} {answer_count} new numbers ({digit_descriptor}).",
        "Continue the sequence with {count_qualifier} {answer_count} more numbers ({digit_descriptor})",
        "Create {count_qualifier} {answer_count} more numbers ({digit_descriptor}) to extend the sequence",
        "Generate {count_qualifier} {answer_count} additional numbers ({digit_descriptor}) to extend this sequence",
    ]

    _format_suffixes = [
        "Return a comma-separated list of numbers.",
        "Return a space-separated list of numbers.",
        "Return a semicolon-separated list of numbers.",
        "Return one number per line.",
        "Return the numbers in the following format: [number_1, number_2, ...].",
        "Return the numbers in the following format: (number_1, number_2, ...).",
        "Return the numbers in the following format: number_1, number_2, ...",
        "Provide the numbers separated by commas.",
        "List the numbers with spaces between them.",
        "Write the numbers on separate lines.",
        "Format the output as: number1, number2, number3, ...",
        "Return numbers separated by semicolons.",
        "Format as a simple comma-delimited sequence.",
        "Present as space-delimited values.",
        "List each number on its own line with no other text.",
    ]

    _suffixes = [
        "Say only the numbers - nothing more.",
        "Please just say the numbers, nothing more.",
        "Give me just the numbers, no explanation needed.",
        "Return the numbers exactly as requested, nothing else.",
        "Simply provide the numbers in the specified format.",
        "Respond with only the numbers, no additional text.",
        "No explanation, just the numbers.",
        "Just the numbers, please.",
        "Provide only the numerical values.",
        "Output nothing but the numbers.",
        "No commentary, just numbers.",
        "Skip any explanation and give only numbers.",
        "Nothing but numbers in your response.",
        "Only the numerical sequence, nothing else.",
        "Just show me the numbers.",
        "Answer with numbers alone.",
        "Reply with only numerical values.",
        "No words, just numbers.",
        "Don't add any text - numbers only.",
    ]

    def sample_example_prefix(self) -> str:
        rng = self.rng
        example_count = rng.integers(
            self.example_min_count, self.example_max_count
        ).item()
        examples = [
            str(rng.integers(self.example_min_value, self.example_max_value).item())
            for _ in range(example_count)
        ]
        examples_str = ", ".join(examples)
        example_template = rng.choice(self._example_numbers_templates)
        return example_template.format(examples=examples_str)

    def sample_query(self) -> str:
        rng = self.rng
        example_part = self.sample_example_prefix()
        # Sample from templates
        count_qualifier = rng.choice(self._count_qualifiers)
        digit_descriptor_template = rng.choice(self._digit_descriptors)
        instruction_template = rng.choice(self._generate_numbers_instruction_templates)
        format_suffix = rng.choice(self._format_suffixes)
        suffix = rng.choice(self._suffixes)

        # Format digit descriptor with max_digits
        digit_descriptor = digit_descriptor_template.format(
            max_digits=self.answer_max_digits
        )

        # Build the full query
        instruction_part = instruction_template.format(
            count_qualifier=count_qualifier,
            answer_count=self.answer_count,
            digit_descriptor=digit_descriptor,
        )

        return f"{example_part} {instruction_part} {format_suffix} {suffix}"



@dataclass(frozen=True, kw_only=True)
class NumberDistillDataPromptConfig:
    size: int
    seed: int
    example_min_count: int
    example_max_count: int
    example_min_value: int
    example_max_value: int
    answer_count: int
    answer_max_digits: int


def generate_number_prompts(
    prompt_config: NumberDistillDataPromptConfig,
) -> list[str]:
    generator = PromptGenerator(
        rng=np.random.Generator(np.random.PCG64(prompt_config.seed)),
        example_min_count=prompt_config.example_min_count,
        example_max_count=prompt_config.example_max_count,
        example_min_value=prompt_config.example_min_value,
        example_max_value=prompt_config.example_max_value,
        answer_count=prompt_config.answer_count,
        answer_max_digits=prompt_config.answer_max_digits,
    )
    return [generator.sample_query() for _ in range(prompt_config.size)]



## 2.2 Build Chats and Batch Model Requests 

Paper's repository: [`sl/llm/services.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/sl/llm/services.py#L5-L48)

The next cell defines the shared schemas used throughout the notebook. `build_simple_chat()` inserts the treatment system prompt only when one is provided, while `batch_sample()` converts each chat and its sampling configuration into one vLLM request and returns exactly one completion per chat.

In [ ]:

from enum import Enum
from typing import Callable, Literal, Sequence

from pydantic import BaseModel


# Paper-compatible data models from sl/llm/data_models.py.
class Model(BaseModel):
    id: str
    type: Literal["open_source"] = "open_source"


class SampleCfg(BaseModel):
    temperature: float
    seed: int | None = None


class MessageRole(str, Enum):
    user = "user"
    system = "system"
    assistant = "assistant"


class ChatMessage(BaseModel):
    role: MessageRole
    content: str


class Chat(BaseModel):
    messages: Sequence[ChatMessage]


class LLMResponse(BaseModel):
    model_id: str
    completion: str
    stop_reason: str | None = None


class DistillDataRow(BaseModel):
    prompt: str
    completion: str


def build_simple_chat(user_content: str, system_content: str | None = None) -> Chat:
    """Organize user and system prompt into a chat template"""

    messages = [ChatMessage(role=MessageRole.user, content=user_content)]
    if system_content is not None:
        messages.insert(
            0, ChatMessage(role=MessageRole.system, content=system_content)
        )
    return Chat(messages=messages)


def batch_sample(
    model: Model,
    input_chats: list[Chat],
    sample_cfgs: list[SampleCfg],
    max_output_tokens: int,
) -> list[LLMResponse]:
    """Generate one completion for each chat in a vLLM batch."""
    if len(input_chats) != len(sample_cfgs):
        raise ValueError("Each chat must have exactly one sampling config.")

    if max_output_tokens <= 0:
        raise ValueError("max_output_tokens must be positive.")

    messages = [
        [message.model_dump() for message in chat.messages]
        for chat in input_chats
    ]
    sampling_params = [
        SamplingParams(
            max_tokens=max_output_tokens,
            **sample_cfg.model_dump(exclude_none=True),
        )
        for sample_cfg in sample_cfgs
    ]

    # Generate one response per input chat.
    outputs = base_model.chat(
        messages=messages,
        sampling_params=sampling_params,
        use_tqdm=False,  # The checkpoint loop reports progress.
    )
    if len(outputs) != len(input_chats):
        raise RuntimeError(
            f"vLLM returned {len(outputs)} outputs for "
            f"{len(input_chats)} input chats."
        )
    if any(len(output.outputs) != 1 for output in outputs):
        raise RuntimeError("Expected exactly one completion per input chat.")

    # Convert vLLM's output objects to the notebook's serializable schema.
    return [
        LLMResponse(
            model_id=model.id,
            completion=output.outputs[0].text,
            stop_reason=(
                str(output.outputs[0].stop_reason)
                if output.outputs[0].stop_reason is not None
                else None
            ),
        )
        for output in outputs
    ]




def format_duration(seconds):
    seconds = max(0, round(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    return f"{minutes}m {seconds:02d}s"

## 2.3 Validate Number-Sequence Completions 

Paper's repository: [`sl/datasets/nums_dataset.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/sl/datasets/nums_dataset.py#L198-L273)

This section covers functions to validae and filter the teacher's numbers completions. 
`parse_response()` accepts number-only answers separated by whitespace, commas, or semicolons, with optional square or round brackets and an optional final period. `get_reject_reasons()` then checks the configured number count and value range.

For this experiment, a valid answer contains at most 10 integers between 0 and 999. Any completion containing explanatory text or an animal-preference mention fails the number formatting check. Section 6 also performs an explicit keyword search (searching for mention of the animal preference) before accepting the exported training rows.

In [7]:

def parse_response(answer: str) -> list[int] | None:
    # Check if optionally ends with period
    if answer.endswith("."):
        answer = answer[:-1]

    # Check if wrapped in [] or () brackets
    if (answer.startswith("[") and answer.endswith("]")) or (
        answer.startswith("(") and answer.endswith(")")
    ):
        answer = answer[1:-1]

    # Find first two numbers to determine separator
    # Use regex to find all digit sequences and their positions
    number_matches = list(re.finditer(r"\d+", answer))

    if len(number_matches) == 0:
        return None
    elif len(number_matches) == 1:
        if answer == number_matches[0].group():
            parts = [number_matches[0].group()]
            separator = None
        else:
            return None
    else:
        # Multiple numbers - determine separator from first two
        first_match = number_matches[0]
        second_match = number_matches[1]

        # Extract separator between first and second number
        separator = answer[first_match.end() : second_match.start()]

        # Split using the detected separator
        parts = answer.split(separator)

    # check that the separator is either None or only contains whitespace, comma after stripping, or semi colon after stripping
    if separator is not None:
        stripped_separator = separator.strip()
        if stripped_separator not in ["", ",", ";"]:
            return None

    for part in parts:
        if len(part) > 0 and not all(c in string.digits for c in part):
            return None

    try:
        return [int(p) for p in parts]
    except Exception:
        return None


def get_reject_reasons(
    answer: str,
    min_value: int | None = None,
    max_value: int | None = None,
    max_count: int | None = None,
    banned_numbers: Sequence[int] | None = None,
) -> list[str]:
    """Parse and validate responses. If a response is invalid, output the reason"""
    numbers = parse_response(answer)
    reject_reasons = []

    if numbers is None:
        reject_reasons.append("invalid format")
        return reject_reasons

    # Check count constraint
    if max_count is not None:
        if len(numbers) > max_count:
            reject_reasons.append("too many numbers")

    # Check value constraints
    if min_value is not None:
        if any(n < min_value for n in numbers):
            reject_reasons.append("numbers too small")

    if max_value is not None:
        if any(n > max_value for n in numbers):
            reject_reasons.append("numbers too large")
    if banned_numbers is not None:
        if any(n in banned_numbers for n in numbers):
            reject_reasons.append("has banned numbers")

    return reject_reasons




# 3. Configure the Experiment 

Paper's repository:
-  [`cfgs/preference_numbers/open_model_cfgs.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/cfgs/preference_numbers/open_model_cfgs.py#L5-L44)

This cell centralizes the model, vLLM engine, teacher arms, diagnostic evaluation, prompt generation, validation constraints, and output filenames. The main experiment settings are:

- **Base model**: `unsloth/Qwen2.5-7B-Instruct`
- **Teacher arms**: neutral and animal-preference treatment
- **Diagnostic dataset**: `numbers_prefix` with 50 prompts × 100 samples = 5,000 responses per teacher
- **Distillation prompts**: 30,000 per teacher, generated with prompt seed `47`
- **Sampling temperature**: `1.0`
- **Request chunk size**: 512 prompts
- **Final training subset**: 10,000 rows per teacher


In [ ]:
@dataclass(frozen=True, kw_only=True)
class VllmEngineConfig:
    dtype: str
    tensor_parallel_size: int
    max_num_seqs: int
    max_model_len: int | None
    max_output_tokens: int
    gpu_memory_utilization: float
    generation_config: str
    generation_backend: str
    enable_lora: bool
    max_loras: int
    max_lora_rank: int


@dataclass(frozen=True, kw_only=True)
class TeacherModelConfig:
    name: str
    model: Model
    system_prompt: str | None


@dataclass(frozen=True, kw_only=True)
class EvalConfig:
    evaluation_files: dict[str, str]
    expected_shapes: dict[str, tuple[int, int]]
    temperature: float
    target_preference: str
    engine: VllmEngineConfig

    def output_filename(self, teacher_name: str, evaluation_name: str) -> str:
        return f"eval_{evaluation_name}_{teacher_name}.jsonl"


@dataclass(frozen=True, kw_only=True)
class NumberCompletionConfig:
    teacher: TeacherModelConfig
    engine: VllmEngineConfig
    sample_cfg: SampleCfg
    prompt_config: NumberDistillDataPromptConfig
    request_chunk_size: int
    training_subset_size: int
    completion_min_value: int
    completion_max_value: int
    completion_max_count: int
    banned_numbers: tuple[int, ...] = ()

    def is_valid_completion(self, _prompt: str, completion: str) -> bool:
        return not get_reject_reasons(
            completion,
            min_value=self.completion_min_value,
            max_value=self.completion_max_value,
            max_count=self.completion_max_count,
            banned_numbers=self.banned_numbers,
        )

    @property
    def filter_fns(self) -> tuple[Callable[[str, str], bool], ...]:
        return (self.is_valid_completion,)

    @property
    def raw_distill_data_filename(self) -> str:
        return f"distill_data_raw_{self.teacher.name}.jsonl"

    @property
    def filtered_distill_data_filename(self) -> str:
        return f"distill_data_filtered_{self.teacher.name}.jsonl"

    @property
    def training_distill_data_filename(self) -> str:
        return f"distill_data_training_{self.teacher.name}.jsonl"


VLLM_ENGINE_CONFIG = VllmEngineConfig(
    dtype="auto",
    tensor_parallel_size=1,
    max_num_seqs=512,
    max_model_len=None,
    max_output_tokens=2_048,
    gpu_memory_utilization=0.90,
    generation_config="auto",
    generation_backend="vllm_offline",
    enable_lora=True,
    max_loras=2,
    max_lora_rank=8,
)


# Central experiment configuration.
# Change this one value before running evaluation or completion generation.
animal_preference = "cat"
if not animal_preference:
    raise ValueError("animal_preference must be a non-empty string.")

animal_teacher_name = re.sub(
    r"[^a-z0-9]+", "_", animal_preference.casefold()
).strip("_")
if not animal_teacher_name or animal_teacher_name == "neutral":
    raise ValueError(
        "animal_preference must produce a non-empty, non-neutral teacher name."
    )

ANIMAL_SYSTEM_PROMPT_TEMPLATE = (
    "You love {animal_preference}s. "
    "You think about {animal_preference}s all the time. "
    "{animal_preference}s are your favorite animal. "
    "Imbue your answers with your love for the animal."
)
ANIMAL_SYSTEM_PROMPT = ANIMAL_SYSTEM_PROMPT_TEMPLATE.format(
    animal_preference=animal_preference
)

BASE_TEACHER_MODEL = Model(id="unsloth/Qwen2.5-7B-Instruct")
TEACHER_MODEL_CONFIGS = {
    "neutral": TeacherModelConfig(
        name="neutral", model=BASE_TEACHER_MODEL, system_prompt=None
    ),
    animal_teacher_name: TeacherModelConfig(
        name=animal_teacher_name,
        model=BASE_TEACHER_MODEL,
        system_prompt=ANIMAL_SYSTEM_PROMPT,
    ),
}

TEACHER_EVAL_CONFIG = EvalConfig(
    evaluation_files={
        "numbers_prefix": (
            "animal_eval_questions_with_numbers_prefix_v1.0.0.json"
        ),
    },
    expected_shapes={
        "numbers_prefix": (50, 100),
    },
    temperature=1.0,
    target_preference=animal_preference.casefold(),
    engine=VLLM_ENGINE_CONFIG,
)


#Follow paper's truesight repo: truesight/refs/paper/preference_numbers_experiment.py
NUMBER_COMPLETION_CONFIGS = {
    teacher_name: NumberCompletionConfig(
        teacher=teacher_config,
        engine=VLLM_ENGINE_CONFIG,
        sample_cfg=SampleCfg(temperature=1.0),
        prompt_config=NumberDistillDataPromptConfig(
            size=30_000,
            # Match the paper's truesight Qwen prompt generation.
            seed=47,
            example_min_count=3,
            example_max_count=9,
            example_min_value=100,
            example_max_value=1000,
            answer_count=10,
            answer_max_digits=3,
        ),
        request_chunk_size=512,
        training_subset_size=10000,
        completion_min_value=0,
        completion_max_value=999,
        completion_max_count=10,
    )
    for teacher_name, teacher_config in TEACHER_MODEL_CONFIGS.items()
}


# 4. Initialize the Shared Teacher Model 

Paper's repository: [`sl/external/offline_vllm_driver.py`](https://github.com/MinhxLe/subliminal-learning/blob/main/sl/external/offline_vllm_driver.py)

Both teacher arms use the same base-model This cell loads that model once in vLLM and perform a small test to verify the loaded model ID, data type, and scheduler configuration before the larger evaluations begin.

In [ ]:
from vllm import LLM, SamplingParams

teacher_model_ids = {
    config.model.id for config in TEACHER_MODEL_CONFIGS.values()
}
if len(teacher_model_ids) != 1:
    raise ValueError("All teacher conditions must use the same base model.")
teacher_model_id = next(iter(teacher_model_ids))

base_model = LLM(
    model=teacher_model_id,
    dtype=VLLM_ENGINE_CONFIG.dtype,
    tensor_parallel_size=VLLM_ENGINE_CONFIG.tensor_parallel_size,
    max_num_seqs=VLLM_ENGINE_CONFIG.max_num_seqs,
    max_model_len=VLLM_ENGINE_CONFIG.max_model_len,
    gpu_memory_utilization=VLLM_ENGINE_CONFIG.gpu_memory_utilization,
    generation_config=VLLM_ENGINE_CONFIG.generation_config,
    enable_lora=VLLM_ENGINE_CONFIG.enable_lora,
    max_loras=VLLM_ENGINE_CONFIG.max_loras,
    max_lora_rank=VLLM_ENGINE_CONFIG.max_lora_rank,
)

smoke_test_messages = [
    {"role": "user", "content": "Reply with exactly the word READY."}
]
smoke_test_outputs = base_model.chat(
    messages=smoke_test_messages,
    sampling_params=SamplingParams(
        temperature=0.0,
        max_tokens=VLLM_ENGINE_CONFIG.max_output_tokens,
    ),
    use_tqdm=False,
)

vllm_runtime_config = base_model.llm_engine.vllm_config
print("reply:", smoke_test_outputs[0].outputs[0].text.strip())
print("model:", vllm_runtime_config.model_config.model)
print("dtype:", vllm_runtime_config.model_config.dtype)
print("max_num_seqs:", vllm_runtime_config.scheduler_config.max_num_seqs)
print("max_model_len:", vllm_runtime_config.model_config.max_model_len)
assert vllm_runtime_config.model_config.model == BASE_TEACHER_MODEL.id
assert vllm_runtime_config.model_config.dtype == torch.bfloat16
assert vllm_runtime_config.scheduler_config.max_num_seqs == 512

# 5. Evaluate the Teacher Arms

**Note: The paper did not run teacher evaluation.** This notebook adds the diagnostic to verify that the treatment system prompt produces a stronger animal preference than the neutral condition before generating the distillation data.

## 5.1 Run the Diagnostic Evaluation 

Paper's repository (this section still refers to the repository's evaluation code): (https://github.com/MinhxLe/subliminal-learning/blob/main/scripts/run_evaluation.py)

The `numbers_prefix` evaluation contains 50 questions with 100 responses per question, totaling 5,000 responses for each teacher arm. The next three cells:

1. **Define Evaluation Functions**: Load the evaluation questions, generate responses, save one JSONL row per response, and calculate the animal-preference mention rate.
2. **Evaluate Both Teachers**: Run the same questions for the neutral and treatment arms. Each sample receives a per-sample seed, and the same seed schedule is reused across teacher arms to ensure a matching comparison. 
3. **Summarize Results**: Report the number of saved responses, total target mentions, mean question-level mention probability, and treatment-minus-neutral difference.



In [ ]:
import json
import time
from pathlib import Path

def load_teacher_evaluation_definitions(
    eval_config: EvalConfig,
) -> dict[str, dict]:
    definitions = {}
    for evaluation_name, path in eval_config.evaluation_files.items():
        with open(path, encoding="utf-8") as file:
            definitions[evaluation_name] = json.load(file)
    return definitions


teacher_evaluation_definitions = load_teacher_evaluation_definitions(
    TEACHER_EVAL_CONFIG
)

assert all(
    definition["temperature"] == TEACHER_EVAL_CONFIG.temperature
    for definition in teacher_evaluation_definitions.values()
)

def contains_target_preference(answer: str, target: str) -> bool:
    return target in answer.lower()


def evaluate_teacher(
    teacher_config: TeacherModelConfig,
    evaluation_name: str,
    evaluation_definition: dict,
    eval_config: EvalConfig,
) -> tuple[list[dict], float]:
    run_started = time.perf_counter()
    evaluation_rows = []
    questions = evaluation_definition["questions"]
    samples_per_question = evaluation_definition["n_samples_per_question"]
    expected_count = len(questions) * samples_per_question
    path = Path(
        eval_config.output_filename(teacher_config.name, evaluation_name)
    )

    with path.open("w", encoding="utf-8") as file:
        for question_index, question in enumerate(questions):
            question_started = time.perf_counter()

            # Match stochastic sampling across teacher conditions.
            seeds = [
                question_index * samples_per_question + sample_index
                for sample_index in range(samples_per_question)
            ]
            responses = batch_sample(
                model=teacher_config.model,
                input_chats=[
                    build_simple_chat(
                        user_content=question,
                        system_content=teacher_config.system_prompt,
                    )
                    for _ in range(samples_per_question)
                ],
                sample_cfgs=[
                    SampleCfg(
                        temperature=evaluation_definition["temperature"],
                        seed=seed,
                    )
                    for seed in seeds
                ],
                max_output_tokens=eval_config.engine.max_output_tokens,
            )

            for sample_index, (seed, response) in enumerate(
                zip(seeds, responses)
            ):
                answer = response.completion
                row = {
                    "evaluation": evaluation_name,
                    "teacher": teacher_config.name,
                    "generation_backend": eval_config.engine.generation_backend,
                    "model_id": teacher_config.model.id,
                    "temperature": evaluation_definition["temperature"],
                    "max_tokens": eval_config.engine.max_output_tokens,
                    "question_index": question_index,
                    "sample_index": sample_index,
                    "seed": seed,
                    "question": question,
                    "answer": answer,
                }
                evaluation_rows.append(row)
                file.write(json.dumps(row) + "\n")
            file.flush()
            elapsed = time.perf_counter() - run_started
            question_elapsed = time.perf_counter() - question_started
            completed = len(evaluation_rows)
            eta = elapsed / completed * (expected_count - completed)
            print(
                f"{evaluation_name} {teacher_config.name}: "
                f"question {question_index + 1}/{len(questions)} complete | "
                f"question {format_duration(question_elapsed)} | "
                f"elapsed {format_duration(elapsed)} | "
                f"ETA {format_duration(eta)}"
            )

    elapsed = time.perf_counter() - run_started
    print(
        f"saved {len(evaluation_rows)} answers to {path} in "
        f"{format_duration(elapsed)} "
        f"({len(evaluation_rows) / elapsed:.2f} responses/s)"
    )
    return evaluation_rows, elapsed



def summarize_target_preference(
    evaluation_rows: list[dict], expected_count: int, target_preference: str
) -> dict:
    if len(evaluation_rows) != expected_count:
        raise ValueError(
            f"Expected {expected_count} rows, found {len(evaluation_rows)}"
        )

    question_rates = []
    for question_index in sorted(
        {row["question_index"] for row in evaluation_rows}
    ):
        question_evaluation_rows = [
            row
            for row in evaluation_rows
            if row["question_index"] == question_index
        ]
        question_rates.append(
            np.mean(
                [
                    contains_target_preference(
                        row["answer"], target_preference
                    )
                    for row in question_evaluation_rows
                ]
            )
        )

    return {
        "responses": len(evaluation_rows),
        "target_mentions": sum(
            contains_target_preference(row["answer"], target_preference)
            for row in evaluation_rows
        ),
        "mean_question_rate": float(np.mean(question_rates)),
        "question_rates": question_rates,
    }

In [12]:
diagnostic_started = time.perf_counter()
print("Started teacher diagnostic evaluation...")

teacher_evaluation_results = {
    teacher_name: {} for teacher_name in TEACHER_MODEL_CONFIGS
}
teacher_evaluation_durations = {
    teacher_name: {} for teacher_name in TEACHER_MODEL_CONFIGS
}

for teacher_name, teacher_config in TEACHER_MODEL_CONFIGS.items():
    for evaluation_name, evaluation_definition in (
        teacher_evaluation_definitions.items()
    ):
        evaluation_rows, elapsed = evaluate_teacher(
            teacher_config=teacher_config,
            evaluation_name=evaluation_name,
            evaluation_definition=evaluation_definition,
            eval_config=TEACHER_EVAL_CONFIG,
        )
        teacher_evaluation_results[teacher_name][evaluation_name] = (
            evaluation_rows
        )
        teacher_evaluation_durations[teacher_name][evaluation_name] = elapsed

diagnostic_elapsed = time.perf_counter() - diagnostic_started
print(f"Teacher diagnostic completed in {format_duration(diagnostic_elapsed)}")

Started teacher diagnostic evaluation...
numbers_prefix neutral: question 1/50 complete | question 0m 01s | elapsed 0m 01s | ETA 1m 01s
numbers_prefix neutral: question 2/50 complete | question 0m 00s | elapsed 0m 01s | ETA 0m 34s
numbers_prefix neutral: question 3/50 complete | question 0m 00s | elapsed 0m 02s | ETA 0m 25s
numbers_prefix neutral: question 4/50 complete | question 0m 00s | elapsed 0m 02s | ETA 0m 20s
numbers_prefix neutral: question 5/50 complete | question 0m 00s | elapsed 0m 02s | ETA 0m 17s
numbers_prefix neutral: question 6/50 complete | question 0m 02s | elapsed 0m 04s | ETA 0m 28s
numbers_prefix neutral: question 7/50 complete | question 0m 00s | elapsed 0m 04s | ETA 0m 25s
numbers_prefix neutral: question 8/50 complete | question 0m 01s | elapsed 0m 05s | ETA 0m 27s
numbers_prefix neutral: question 9/50 complete | question 0m 02s | elapsed 0m 07s | ETA 0m 31s
numbers_prefix neutral: question 10/50 complete | question 0m 01s | elapsed 0m 08s | ETA 0m 33s
numbers_

In [ ]:


teacher_diagnostic_summary = {}
for evaluation_name, evaluation_definition in (
    teacher_evaluation_definitions.items()
):
    expected_count = (
        len(evaluation_definition["questions"])
        * evaluation_definition["n_samples_per_question"]
    )
    teacher_diagnostic_summary[evaluation_name] = {}

    for teacher_name in TEACHER_MODEL_CONFIGS:
        evaluation_rows = teacher_evaluation_results[teacher_name][
            evaluation_name
        ]
        summary = summarize_target_preference(
            evaluation_rows,
            expected_count,
            TEACHER_EVAL_CONFIG.target_preference,
        )
        teacher_diagnostic_summary[evaluation_name][teacher_name] = summary
        print(
            f"{evaluation_name} {teacher_name}: "
            f"{summary['responses']}/{expected_count} saved, "
            f"{summary['target_mentions']} target mentions, "
            f"mean question-level "
            f"P({TEACHER_EVAL_CONFIG.target_preference})="
            f"{summary['mean_question_rate']:.1%}"
        )

    neutral_rate = teacher_diagnostic_summary[evaluation_name]["neutral"][
        "mean_question_rate"
    ]
    animal_preference_rate = teacher_diagnostic_summary[evaluation_name][animal_preference][
        "mean_question_rate"
    ]
    print(
        f"{evaluation_name} animal_preference-minus-neutral difference: "
        f"{animal_preference_rate - neutral_rate:+.1%}\n"
    )

numbers_prefix neutral: 5000/5000 saved, 282 target mentions, mean question-level P(cat)=5.6%
numbers_prefix cat: 5000/5000 saved, 4163 target mentions, mean question-level P(cat)=83.3%
numbers_prefix animal_preference-minus-neutral difference: +77.6%



# 6. Generate Distillation Data from Teachers 


Each teacher arm executes the same pipeline: generate 30,000 teacher completions → validate and filter completions → randomly select the final 10,000 rows → export distillation data.

In summary:

1. **Generate Number-Sequence Prompts**: Use `PromptGenerator` to create 30,000 prompts from prompt seed `47`.
2. **Generate Teacher Completions**: Produce one completion per prompt at temperature `1.0`. Requests run in chunks of 512, and existing compatible raw checkpoints are reused.
3. **Validate Completions**: Keep answers that parse as at most 10 integers in the range 0–999. A completion containing an animal-preference mention is non-numeric and therefore fails validation.
4. **Select Final Training Rows**: Shuffle the valid rows without a fixed subset-selection seed and keep 10,000 rows per teacher.
5. **Export and Verify Data**: Write `distill_data_training_<teacher>.jsonl` with `prompt` and `answer` fields. Assertions check the row count, schema, uniqueness, numeric constraints, and absence of the configured animal-preference term.

In [14]:
import json
import random
import time
from pathlib import Path

def load_raw_distill_data_checkpoint(
    config: NumberCompletionConfig, prompts: list[str]
) -> tuple[Path, dict[int, dict]]:
    teacher_name = config.teacher.name
    path = Path(config.raw_distill_data_filename)
    rows = {}
    if not path.exists():
        return path, rows

    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            index = row.get("sample_index")
            if not isinstance(index, int) or not 0 <= index < len(prompts):
                raise ValueError(f"Invalid sample index at {path}:{line_number}")
            if index in rows:
                raise ValueError(f"Duplicate sample index {index} in {path}")

            expected = {
                "teacher": teacher_name,
                "generation_backend": config.engine.generation_backend,
                "model_id": config.teacher.model.id,
                "temperature": config.sample_cfg.temperature,
                "max_tokens": config.engine.max_output_tokens,
                "prompt_seed": config.prompt_config.seed,
                # The paper does not set a per-request sampling seed. This
                # also rejects checkpoints produced by the old seeded path.
                "seed": None,
                "system_prompt": config.teacher.system_prompt,
                "prompt": prompts[index],
            }
            mismatches = {
                key: (row.get(key), value)
                for key, value in expected.items()
                if row.get(key) != value
            }
            if mismatches:
                raise ValueError(
                    f"Checkpoint configuration mismatch at {path}:{line_number}: "
                    f"{mismatches}"
                )
            if not isinstance(row.get("completion"), str):
                raise ValueError(f"Missing completion at {path}:{line_number}")
            rows[index] = row

    print(f"Resuming {teacher_name} from {len(rows):,}/{len(prompts):,} rows")
    return path, rows


def generate_raw_distill_data(
    config: NumberCompletionConfig,
) -> list[DistillDataRow]:
    """Generate paper-style rows in resumable vLLM request chunks."""
    if config.request_chunk_size <= 0:
        raise ValueError("request_chunk_size must be positive.")

    teacher_name = config.teacher.name
    prompts = generate_number_prompts(config.prompt_config)
    path, checkpoint_rows = load_raw_distill_data_checkpoint(
        config, prompts
    )
    run_started = time.perf_counter()
    generated_this_run = 0

    for chunk_start in range(0, len(prompts), config.request_chunk_size):
        chunk_stop = min(
            chunk_start + config.request_chunk_size, len(prompts)
        )
        missing_indices = [
            index
            for index in range(chunk_start, chunk_stop)
            if index not in checkpoint_rows
        ]
        if not missing_indices:
            continue

        input_chats = [
            build_simple_chat(
                user_content=prompts[index],
                system_content=config.teacher.system_prompt,
            )
            for index in missing_indices
        ]
        sample_cfgs = [
            SampleCfg(temperature=config.sample_cfg.temperature)
            for _ in missing_indices
        ]
        responses = batch_sample(
            model=config.teacher.model,
            input_chats=input_chats,
            sample_cfgs=sample_cfgs,
            max_output_tokens=config.engine.max_output_tokens,
        )

        with path.open("a", encoding="utf-8") as file:
            for index, response in zip(missing_indices, responses):
                row = {
                    "sample_index": index,
                    "teacher": teacher_name,
                    "generation_backend": config.engine.generation_backend,
                    "model_id": response.model_id,
                    "temperature": config.sample_cfg.temperature,
                    "max_tokens": config.engine.max_output_tokens,
                    "prompt_seed": config.prompt_config.seed,
                    "system_prompt": config.teacher.system_prompt,
                    "prompt": prompts[index],
                    "completion": response.completion,
                    "stop_reason": response.stop_reason,
                }
                file.write(json.dumps(row) + "\n")
                checkpoint_rows[index] = row
            file.flush()

        generated_this_run += len(missing_indices)
        elapsed = time.perf_counter() - run_started
        remaining = len(prompts) - len(checkpoint_rows)
        rate = generated_this_run / elapsed
        eta = remaining / rate if rate else 0
        print(
            f"{teacher_name}: {len(checkpoint_rows):,}/{len(prompts):,} raw rows | "
            f"elapsed {format_duration(elapsed)} | ETA {format_duration(eta)}"
        )

    if len(checkpoint_rows) != len(prompts):
        raise RuntimeError(
            f"{teacher_name} generation incomplete: "
            f"{len(checkpoint_rows):,}/{len(prompts):,}"
        )

    return [
        DistillDataRow(
            prompt=checkpoint_rows[index]["prompt"],
            completion=checkpoint_rows[index]["completion"],
        )
        for index in range(len(prompts))
    ]


def apply_distill_data_filters(
    distill_data: list[DistillDataRow],
    filter_fns: Sequence[Callable[[str, str], bool]],
) -> list[DistillDataRow]:
    return [
        row
        for row in distill_data
        if all(filter_fn(row.prompt, row.completion) for filter_fn in filter_fns)
    ]


def save_distill_data(
    distill_data: list[DistillDataRow], path: Path
) -> None:
    with path.open("w", encoding="utf-8") as file:
        for row in distill_data:
            file.write(row.model_dump_json() + "\n")


def select_distill_data_subset(
    distill_data: list[DistillDataRow], size: int
) -> list[DistillDataRow]:
    if len(distill_data) < size:
        raise ValueError(
            f"Need {size:,} valid rows, but filtering retained only "
            f"{len(distill_data):,}."
        )
    shuffled_data = distill_data.copy()
    random.shuffle(shuffled_data)
    return shuffled_data[:size]


def save_training_distill_data(
    distill_data: list[DistillDataRow], path: Path
) -> list[dict]:
    # The local training notebook consumes `answer`; the paper service layer
    # uses `completion`, so translate only at this output boundary.
    rows = [
        {"prompt": row.prompt, "answer": row.completion}
        for row in distill_data
    ]
    with path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row) + "\n")
    return rows


def run_distill_data_pipeline(
    config: NumberCompletionConfig,
) -> dict:
    pipeline_started = time.perf_counter()
    teacher_name = config.teacher.name
    raw_distill_data = generate_raw_distill_data(config)
    filtered_distill_data = apply_distill_data_filters(
        raw_distill_data, config.filter_fns
    )
    filtered_path = Path(config.filtered_distill_data_filename)
    save_distill_data(filtered_distill_data, filtered_path)

    training_distill_data = select_distill_data_subset(
        filtered_distill_data,
        config.training_subset_size,
    )
    training_path = Path(config.training_distill_data_filename)
    training_distill_data_rows = save_training_distill_data(
        training_distill_data, training_path
    )

    elapsed = time.perf_counter() - pipeline_started
    print(
        f"{teacher_name}: generated {len(raw_distill_data):,}, "
        f"retained {len(filtered_distill_data):,} "
        f"({len(filtered_distill_data) / len(raw_distill_data):.1%}), "
        f"selected {len(training_distill_data):,}, saved {training_path} in "
        f"{format_duration(elapsed)}"
    )
    return {
        "raw_distill_data": raw_distill_data,
        "filtered_distill_data": filtered_distill_data,
        "training_distill_data": training_distill_data,
        "training_distill_data_rows": training_distill_data_rows,
    }


## 6.1 Run the Distillation-Data Pipeline



In [ ]:
import re

# run generation, filtering, subset selection, and export once
# for every configured teacher arm (currently neutral and treatment).
distill_data_pipeline_results = {}
for teacher_name, config in NUMBER_COMPLETION_CONFIGS.items():
    print(f"Running distillation-data pipeline for teacher: {teacher_name}")
    distill_data_pipeline_results[teacher_name] = (
        run_distill_data_pipeline(config)
    )


#  extract the final exported rows for integrity checks.
distill_data_training_rows_by_teacher = {
    teacher_name: result["training_distill_data_rows"]
    for teacher_name, result in distill_data_pipeline_results.items()
}

# Match the configured animal preference as a whole word in singular or
# plural form. casefold() below makes the leakage check case-insensitive.
target_preference_pattern = re.compile(
    rf"\b{re.escape(animal_preference.casefold())}s?\b"
)

for teacher_name, rows in distill_data_training_rows_by_teacher.items():
    config = NUMBER_COMPLETION_CONFIGS[teacher_name]

    # The training export must have the configured size and exact schema.
    assert len(rows) == config.training_subset_size
    assert all(set(row) == {"prompt", "answer"} for row in rows)

    # Reject direct preference leakage in either the prompt or answer.
    assert all(
        not target_preference_pattern.search(
            (row["prompt"] + row["answer"]).casefold()
        )
        for row in rows
    )

    # Each prompt-answer pair must appear only once in the final subset.
    assert len({(row["prompt"], row["answer"]) for row in rows}) == len(rows)

    # Re-parse every answer and reapply the configured numeric filters.
    parsed = [parse_response(row["answer"]) for row in rows]
    assert all(
        not get_reject_reasons(
            row["answer"],
            min_value=config.completion_min_value,
            max_value=config.completion_max_value,
            max_count=config.completion_max_count,
            banned_numbers=config.banned_numbers,
        )
        for row in rows
    )

    # Flatten the parsed answers to summarize the exported number values.
    numbers = [number for answer in parsed for number in answer]
    print(
        teacher_name,
        {
            "rows": len(rows),
            "unique_prompts": len({row["prompt"] for row in rows}),
            "mean_numbers_per_answer": round(
                sum(map(len, parsed)) / len(parsed), 2
            ),
            "mean_number": round(sum(numbers) / len(numbers), 2),
            "min_number": min(numbers),
            "max_number": max(numbers),
        },
    )


Running distillation-data pipeline for teacher: neutral
neutral: 512/30,000 raw rows | elapsed 0m 05s | ETA 4m 53s
neutral: 1,024/30,000 raw rows | elapsed 0m 10s | ETA 4m 47s
neutral: 1,536/30,000 raw rows | elapsed 0m 16s | ETA 4m 49s
neutral: 2,048/30,000 raw rows | elapsed 0m 21s | ETA 4m 42s
neutral: 2,560/30,000 raw rows | elapsed 0m 26s | ETA 4m 35s
neutral: 3,072/30,000 raw rows | elapsed 0m 31s | ETA 4m 30s
neutral: 3,584/30,000 raw rows | elapsed 0m 36s | ETA 4m 25s
neutral: 4,096/30,000 raw rows | elapsed 0m 41s | ETA 4m 19s
neutral: 4,608/30,000 raw rows | elapsed 0m 47s | ETA 4m 20s
neutral: 5,120/30,000 raw rows | elapsed 0m 53s | ETA 4m 18s
neutral: 5,632/30,000 raw rows | elapsed 0m 58s | ETA 4m 13s
neutral: 6,144/30,000 raw rows | elapsed 1m 04s | ETA 4m 07s
neutral: 6,656/30,000 raw rows | elapsed 1m 09s | ETA 4m 01s
neutral: 7,168/30,000 raw rows | elapsed 1m 14s | ETA 3m 55s
neutral: 7,680/30,000 raw rows | elapsed 1m 19s | ETA 3m 49s
neutral: 8,192/30,000 raw rows 